In [46]:
import pandas as pd
from pathlib import Path

def fetch_data():
    cols = ["pl_name", "hostname", "discoverymethod", "pl_controv_flag", "pl_rade", "pl_bmasse", "pl_orbper", "pl_orbsmax", "pl_orbeccen", 
            "pl_insol", "pl_eqt", "st_teff", "st_rad", "st_mass", "st_lum"]  # all needed collumns
    query = "select " + ",".join(cols) + " from pscomppars" #allows selecting each collumnn from the table
    url = ("https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query="
        + query.replace(" ", "+") + "&format=csv")

    path = Path("data/planets_raw.csv")
    if path.exists():
        df = pd.read_csv(path)
        print(f"Loaded {len(df)} planets from cache")
    else:
        df = pd.read_csv(url)
        path.parent.mkdir(exist_ok=True)
        df.to_csv(path, index=False)
        print(f"Downloaded {len(df)} planets from NASA")


    return df

def fill_derived(df):

    #calculates incase missing before insolation calculations
    mask = df["pl_orbsmax"].isna() & df["st_mass"].notna() & df["pl_orbper"].notna() #finds where orbdistance is absent but can be calculated
    df.loc[mask, "pl_orbsmax"] = ((df.loc[mask, "st_mass"])*(((df.loc[mask, "pl_orbper"])/365.25)**2))**(1/3) #kepler's law
    df["pl_orbsmax_derived"]=mask #states if derived

    mask = df["pl_insol"].isna() & df["st_lum"].notna() & df["pl_orbsmax"].notna() #finds where insolation is absent but it can also be calculated
    df.loc[mask, "pl_insol"] = 10 ** df.loc[mask, "st_lum"] / df.loc[mask, "pl_orbsmax"] ** 2 #update only stated rows with calculated insol
    df["pl_insol_derived"] = mask #clarifies if we derived it or not

    print(f"Filled {df['pl_orbsmax_derived'].sum()} distances, {df['pl_insol_derived'].sum()} insolations")
    return df

def elim_bad_rows(df):
    bad = (df["pl_controv_flag"] == 1) | (df["pl_rade"] <= 0) | (df["pl_orbeccen"] >= 1) #removes controversial or impossible planets
    print("Removing", bad.sum(), "planets") #Removing 58 planets
    df = df[~bad].reset_index(drop=True)
    return df

In [47]:
if __name__ == "__main__":
    df = fetch_data()
    df = fill_derived(df)
    df = elim_bad_rows(df)
    df.to_csv("data/planets_cleaned.csv", index=False)
    print("Saved", len(df), "planets")
    
'''
(df.isna().mean() * 100).sort_values(ascending=False) #finds percentage each collumn is empty for
test = pd.DataFrame({"pl_orbper": [365.25], "st_mass": [1.0], "st_lum": [0.0],
                     "pl_orbsmax": [float("nan")], "pl_insol": [float("nan")]})
fill_derived(df)
(df.isna().mean() * 100).sort_values(ascending=False) #finds percentage each collumn is empty for

print(df["pl_orbsmax_derived"].sum(), "distances filled")   #491 distances filled
print(df["pl_insol_derived"].sum(), "insolations filled")   #269 insolations filled
'''



Query: select pl_name,hostname,discoverymethod,pl_controv_flag,pl_rade,pl_bmasse,pl_orbper,pl_orbsmax,pl_orbeccen,pl_insol,pl_eqt,st_teff,st_rad,st_mass,st_lum from pscomppars
URL: https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+pl_name,hostname,discoverymethod,pl_controv_flag,pl_rade,pl_bmasse,pl_orbper,pl_orbsmax,pl_orbeccen,pl_insol,pl_eqt,st_teff,st_rad,st_mass,st_lum+from+pscomppars&format=csv
(6464, 15)
Removing 58 planets
Saved 6406 planets


'\n(df.isna().mean() * 100).sort_values(ascending=False) #finds percentage each collumn is empty for\ntest = pd.DataFrame({"pl_orbper": [365.25], "st_mass": [1.0], "st_lum": [0.0],\n                     "pl_orbsmax": [float("nan")], "pl_insol": [float("nan")]})\nfill_derived(df)\n(df.isna().mean() * 100).sort_values(ascending=False) #finds percentage each collumn is empty for\n\nprint(df["pl_orbsmax_derived"].sum(), "distances filled")   #491 distances filled\nprint(df["pl_insol_derived"].sum(), "insolations filled")   #269 insolations filled\n'

0 distances filled
0 insolations filled


Removing 58 planets


In [85]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/planets_cleaned.csv")
hz_coeffs = {
    "recent_venus":       (1.776, 2.136e-4, 2.533e-8, -1.332e-11, -3.097e-15),  # optimistic inner
    "runaway_greenhouse": (1.107, 1.332e-4, 1.580e-8, -8.308e-12, -1.931e-15),  # conservative inner
    "max_greenhouse":     (0.356, 6.171e-5, 1.698e-9, -3.198e-12, -5.575e-16),  # conservative outer
    "early_mars":         (0.320, 5.547e-5, 1.526e-9, -2.874e-12, -5.011e-16),  # optimistic outer
}

def hz_edge (starTemp, edgeType):
    solarV, a, b, c, d = hz_coeffs[edgeType]
    normalized_temp=starTemp-5780
    limit = solarV + a*(normalized_temp) + b*(normalized_temp**2) + c*(normalized_temp**3) + d*(normalized_temp**4)
    return limit

def add_hz_edgeData(df):

    teff_used = df["st_teff"].clip(2600, 7200)

    for edgeType in hz_coeffs:
        df[f"hz_{edgeType}"] = hz_edge(teff_used, edgeType)

    #Kopparapu formula only works for these star temps (https://ui.adsabs.harvard.edu/abs/2014ApJ...787L..29K/abstract)
        
    df["not_og_teff"] = ~df["st_teff"].between(2600, 7200) & df["st_teff"].notna()

    valid = df["st_teff"].between(2500, 7500) #very few planets would be considered fully with a change of range, only Trappist d,e,f,g (outside limit by 34 k, passes required tests with this change)

    df["within_conservative"] = (valid & (df["hz_runaway_greenhouse"] >= df["pl_insol"]) & (df["hz_max_greenhouse"] <= df["pl_insol"]))
    df["within_optimistic"] = (valid & (df["hz_recent_venus"] >= df["pl_insol"]) & (df["hz_early_mars"] <= df["pl_insol"]))

    #at initial execution: 181 planets in conservative HZ, 280 in optimistic HZ
    print(f"{df['within_conservative'].sum()} planets in conservative HZ, "
          f"{df['within_optimistic'].sum()} in optimistic HZ")
    print(f"{df['not_og_teff'].sum()} stars outside the calibrated range (clamped)")

    return df

def classify_type(df):
    df["pl_type"] = pd.cut(
        df["pl_rade"],
        bins=[0, 1.5, 2.0, np.inf],
        labels=["rocky", "uncertain", "gaseous"],
    )
    return df

df = add_hz_edgeData(df)
df = classify_type(df)
df["pl_type"].value_counts()

def find_candidates(df):
    mask = df["within_optimistic"] & df["pl_type"].isin(["rocky", "uncertain"])
    cols = ["pl_name", "pl_rade", "pl_insol", "st_teff",
            "within_conservative", "pl_type", "pl_bmassprov"]
    return df.loc[mask, cols].sort_values(["within_conservative", "pl_rade"], ascending=[False, True])   

known = ["TRAPPIST-1 e", "TRAPPIST-1 f", "TRAPPIST-1 g", "Kepler-442 b",
         "Proxima Cen b", "TOI-700 d", "LHS 1140 b", "Teegarden's Star b"]

candidates = find_candidates(df)
# --- Investigation: which stars were clamped, and does it affect any candidates? ---
df["is_candidate"] = df["pl_name"].isin(candidates["pl_name"])
known = ["TRAPPIST-1 e", "TRAPPIST-1 f", "TRAPPIST-1 g", "Kepler-442 b",
         "Proxima Cen b", "TOI-700 d", "LHS 1140 b", "Teegarden's Star b"]

candidates = find_candidates(df)
for name in known:
    found = name in candidates["pl_name"].values
    print(f"{name:22} {'FOUND' if found else 'MISSING'}")
clamped_stars = (
    df[df["not_og_teff"]]
    .groupby("hostname")
    .agg(st_teff=("st_teff", "first"),
         n_planets=("pl_name", "count"),
         n_candidates=("is_candidate", "sum"))
    .sort_values("st_teff")
)
clamped_stars["off_by_K"] = np.where(clamped_stars["st_teff"] < 2600,
                                     2600 - clamped_stars["st_teff"],
                                     clamped_stars["st_teff"] - 7200)

stretched = df[
    df["not_og_teff"]
    & df["st_teff"].between(2300, 7500)
    & (df["pl_type"] != "gaseous")
]
cols = ["pl_name", "st_teff", "pl_rade", "pl_type", "pl_insol",
        "within_conservative", "within_optimistic"]
print(f"{len(stretched)} non-gaseous planets around stretched stars")
print(stretched[cols].sort_values("st_teff").to_string(index=False))

184 planets in conservative HZ, 284 in optimistic HZ
70 stars outside the calibrated range (clamped)
TRAPPIST-1 e           FOUND
TRAPPIST-1 f           FOUND
TRAPPIST-1 g           FOUND
Kepler-442 b           FOUND
Proxima Cen b          FOUND
TOI-700 d              FOUND
LHS 1140 b             FOUND
Teegarden's Star b     FOUND
9 non-gaseous planets around stretched stars
     pl_name  st_teff  pl_rade pl_type  pl_insol  within_conservative  within_optimistic
TRAPPIST-1 b   2566.0    1.116   rocky    4.1530                False              False
TRAPPIST-1 c   2566.0    1.097   rocky    2.2140                False              False
TRAPPIST-1 d   2566.0    0.788   rocky    1.1150                False               True
TRAPPIST-1 e   2566.0    0.920   rocky    0.6460                 True               True
TRAPPIST-1 f   2566.0    1.045   rocky    0.3730                 True               True
TRAPPIST-1 g   2566.0    1.129   rocky    0.2520                 True               True

In [86]:
import pandas as pd

candidates = pd.read_csv("data/candidates.csv")
print(len(candidates), "candidates\n")
print(candidates["within_conservative"].value_counts(), "\n")
print(candidates["pl_type"].value_counts(), "\n")
print(candidates["pl_bmassprov"].value_counts(), "\n")
print(candidates.head(15).to_string(index=False))

45 candidates

within_conservative
True     27
False    18
Name: count, dtype: int64 

pl_type
rocky        25
uncertain    20
Name: count, dtype: int64 

pl_bmassprov
M-R relationship    17
Msini               16
Mass                12
Name: count, dtype: int64 

           pl_name  pl_rade  pl_insol  st_teff  within_conservative pl_type     pl_bmassprov
      TRAPPIST-1 e    0.920     0.646   2566.0                 True   rocky             Mass
Teegarden's Star c    1.020     0.350   3034.0                 True   rocky            Msini
     Proxima Cen b    1.020     0.641   2900.0                 True   rocky            Msini
         GJ 1002 b    1.030     0.670   3024.0                 True   rocky            Msini
      TRAPPIST-1 f    1.045     0.373   2566.0                 True   rocky             Mass
     Kepler-1649 c    1.060     0.750   3240.0                 True   rocky M-R relationship
       Wolf 1069 b    1.080     0.652   3158.0                 True   rocky         

In [93]:
from habitable import add_atmo_score

test = pd.DataFrame({
    "pl_name":   ["Earth", "Mars"],
    "pl_bmasse": [1.0, 0.107],
    "pl_rade":   [1.0, 0.532],
    "pl_insol":  [1.0, 0.43],
})
print(add_atmo_score(test)[["pl_name", "esc_vel", "atmo_score", "atmo_class"]])

  pl_name   esc_vel  atmo_score   atmo_class
0   Earth  0.500000    0.062500     marginal
1    Mars  0.100564    0.000238  likely_lost


In [94]:
import numpy as np
import pandas as pd

HZ_COEFFS = {
    "recent_venus":       (1.776, 2.136e-4, 2.533e-8, -1.332e-11, -3.097e-15),  # optimistic inner
    "runaway_greenhouse": (1.107, 1.332e-4, 1.580e-8, -8.308e-12, -1.931e-15),  # conservative inner
    "max_greenhouse":     (0.356, 6.171e-5, 1.698e-9, -3.198e-12, -5.575e-16),  # conservative outer
    "early_mars":         (0.320, 5.547e-5, 1.526e-9, -2.874e-12, -5.011e-16),  # optimistic outer
}

KNOWN_CANDIDATES = ["TRAPPIST-1 e", "TRAPPIST-1 f", "TRAPPIST-1 g", "Kepler-442 b",
                    "Proxima Cen b", "TOI-700 d", "LHS 1140 b", "Teegarden's Star b"]

def hz_edge (starTemp, edgeType):
    solarV, a, b, c, d = HZ_COEFFS[edgeType]
    normalized_temp=starTemp-5780
    limit = solarV + a*(normalized_temp) + b*(normalized_temp**2) + c*(normalized_temp**3) + d*(normalized_temp**4)
    return limit

def add_hz_edgeData(df):

    #Kopparapu formula only works for these star temps (https://ui.adsabs.harvard.edu/abs/2014ApJ...787L..29K/abstract)
    teff_used = df["st_teff"].clip(2600, 7200)

    for edgeType in HZ_COEFFS:
        df[f"hz_{edgeType}"] = hz_edge(teff_used, edgeType)

    df["not_og_teff"] = ~df["st_teff"].between(2600, 7200) & df["st_teff"].notna()

    #adjusts for planets slightly outside range
    valid = df["st_teff"].between(2500, 7500) #very few planets would be considered fully with a change of range, only Trappist d,e,f,g (outside limit by 34 k, passes required tests with this change)
    #upper limit unchanged as it has no effect
    df["within_conservative"] = (valid & (df["hz_runaway_greenhouse"] >= df["pl_insol"]) & (df["hz_max_greenhouse"] <= df["pl_insol"]))
    df["within_optimistic"] = (valid & (df["hz_recent_venus"] >= df["pl_insol"]) & (df["hz_early_mars"] <= df["pl_insol"]))

    #at initial execution: 184 planets in conservative HZ, 284 in optimistic HZ
    print(f"{df['within_conservative'].sum()} planets in conservative HZ, "
          f"{df['within_optimistic'].sum()} in optimistic HZ")
    print(f"{df['not_og_teff'].sum()} planets outside the initial calibrated range")

    return df

def classify_type(df):
    df["pl_type"] = pd.cut(
        df["pl_rade"],
        bins=[0, 1.5, 2.0, np.inf],
        labels=["rocky", "uncertain", "gaseous"],
    )
    return df

def add_data_quality(df):
    # Mass: radius and mass measured, high trust, M-R: radius measured, mass estimated from radius, medium trust: size is real mass is a guess
    #Msini = radius was estimated from mass, not measured, lowest trust, size is a guess
    df["data_quality"] = df["pl_bmassprov"].map({
        "Mass": "high",
        "M-R relationship": "medium",
        "Msini": "low",
    })
    return df

#planet keeps its atmosphere if its gravity beats the energy its star pours in
def add_atmo_score(df):
    df["esc_vel"] = (df["pl_bmasse"] / df["pl_rade"])**(1/2)
    df["atmo_score"]= (df["esc_vel"]**4) / df["pl_insol"]

    df["atmo_class"] = pd.cut(
        df["atmo_score"],
        bins=[0, 0.01, 0.1, np.inf],
        labels=["likely_lost", "marginal", "likely_retained"],
    )
    return df

def find_candidates(df):
    mask = df["within_optimistic"] & df["pl_type"].isin(["rocky", "uncertain"])
    cols = ["pl_name", "pl_rade", "pl_insol", "st_teff",
            "within_conservative", "pl_type", "pl_bmassprov", "data_quality", "atmo_score", "atmo_class"]
    return df.loc[mask, cols].sort_values(["within_conservative", "pl_rade" ], ascending=[False, True])   

def check_known(candidates):
    #verifies known candidates that could be lost, are not 
    for name in KNOWN_CANDIDATES:
        found = name in candidates["pl_name"].values
        print(f"{name:22} {'FOUND' if found else 'MISSING'}")

if __name__ == "__main__":

    df = pd.read_csv("data/planets_cleaned.csv")

    df = add_hz_edgeData(df)
    df = classify_type(df)
    df = add_data_quality(df)
    df = add_atmo_score(df)
    candidates = find_candidates(df)
    
    print(f"\n{len(candidates)} candidates (optimistic HZ, rocky or uncertain size)\n")
    check_known(candidates)

    df.to_csv("data/planets_scored.csv", index=False)
    candidates.to_csv("data/candidates.csv", index=False)

    

184 planets in conservative HZ, 284 in optimistic HZ
70 planets outside the initial calibrated range

45 candidates (optimistic HZ, rocky or uncertain size)

TRAPPIST-1 e           FOUND
TRAPPIST-1 f           FOUND
TRAPPIST-1 g           FOUND
Kepler-442 b           FOUND
Proxima Cen b          FOUND
TOI-700 d              FOUND
LHS 1140 b             FOUND
Teegarden's Star b     FOUND


In [95]:
test = pd.DataFrame({
    "pl_name":   ["Earth", "Mars"],
    "pl_bmasse": [1.0, 0.107],
    "pl_rade":   [1.0, 0.532],
    "pl_insol":  [1.0, 0.43],
})
print(add_atmo_score(test)[["pl_name", "esc_vel", "atmo_score", "atmo_class"]])

  pl_name   esc_vel  atmo_score       atmo_class
0   Earth  1.000000    1.000000  likely_retained
1    Mars  0.448473    0.094075         marginal


In [3]:
import pandas as pd

df = pd.read_csv("data/planets_scored.csv")
candidate_names = pd.read_csv("data/candidates.csv")["pl_name"]
cand = df[df["pl_name"].isin(candidate_names)]

err_cols = ["pl_radeerr1", "pl_insolerr1", "st_tefferr1", "pl_bmasseerr1"]
print(cand[err_cols].notna().mean() * 100)

pl_radeerr1      57.777778
pl_insolerr1     91.111111
st_tefferr1      97.777778
pl_bmasseerr1    53.333333
dtype: float64


In [4]:
print(pd.crosstab(cand["data_quality"], cand["pl_radeerr1"].isna(), margins=True))
print(pd.crosstab(cand["data_quality"], cand["pl_bmasseerr1"].isna(), margins=True))

pl_radeerr1   False  True  All
data_quality                  
high              9     3   12
low               0    16   16
medium           17     0   17
All              26    19   45
pl_bmasseerr1  False  True  All
data_quality                   
high               8     4   12
low               16     0   16
medium             0    17   17
All               24    21   45


In [5]:
import pandas as pd

mc = pd.read_csv("data/planets_mc.csv")
cand_names = pd.read_csv("data/candidates.csv")["pl_name"]
cand = mc[mc["pl_name"].isin(cand_names)]

print("Candidates:", len(cand))
print("Conservative zone:", cand["within_conservative"].sum())
print(cand["data_quality"].value_counts(), "\n")

for threshold in [0.9, 0.5]:
    print(f"p_strong >= {threshold:.0%}:", (mc["p_strong"] >= threshold).sum())

print("\nKepler-440 b p_candidate:", round(mc.loc[mc["pl_name"] == "Kepler-440 b", "p_candidate"].iloc[0], 2))
print("\nTop 10:")
print(mc.sort_values(["p_strong", "p_candidate"], ascending=False)
        [["pl_name", "p_strong", "pl_rade", "pl_insol", "st_teff", "data_quality"]]
        .head(10).round(2).to_string(index=False))

Candidates: 45
Conservative zone: 27
data_quality
low       19
medium    17
high       9
Name: count, dtype: int64 

p_strong >= 90%: 8
p_strong >= 50%: 15

Kepler-440 b p_candidate: 0.52

Top 10:
           pl_name  p_strong  pl_rade  pl_insol  st_teff data_quality
      TRAPPIST-1 e      0.99     0.92      0.65   2566.0         high
      TRAPPIST-1 f      0.99     1.04      0.37   2566.0         high
      TRAPPIST-1 g      0.99     1.13      0.25   2566.0         high
     Proxima Cen b      0.94     1.02      0.64   2900.0          low
Teegarden's Star c      0.94     1.02      0.35   3034.0          low
         GJ 1002 b      0.93     1.03      0.67   3024.0          low
       Wolf 1069 b      0.90     1.08      0.65   3158.0          low
       Kepler-62 f      0.90     1.41      0.50   4925.0         high
     Kepler-1649 c      0.85     1.06      0.75   3240.0       medium
         GJ 1002 c      0.83     1.10      0.26   3024.0          low
